# 20.8 Monitoring and drift detection

Monitoring asks whether the world still matches the training contract: features, predictions, labels, and business outcomes are all time series.

Feature stores provide distributions to compare and registries identify the model that produced a live stream. Here we compute PSI, z-shift, and operational alerts across monitoring workloads.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2008
rng = np.random.default_rng(SEED)


def drift_stats(base_probs, live_probs, base_values=None, live_values=None):
    eps = 1e-9
    base_probs = np.asarray(base_probs, dtype=float)
    live_probs = np.asarray(live_probs, dtype=float)
    psi_terms = (live_probs - base_probs) * np.log((live_probs + eps) / (base_probs + eps))
    psi = float(np.sum(psi_terms))
    z_shift = np.nan
    if base_values is not None and live_values is not None:
        base_values = np.asarray(base_values, dtype=float)
        live_values = np.asarray(live_values, dtype=float)
        z_shift = float((np.mean(live_values) - np.mean(base_values)) / np.std(base_values, ddof=0))
    return {"psi": psi, "psi_terms": psi_terms, "z_shift": z_shift}


def histogram_probs(values, bins):
    counts, _ = np.histogram(values, bins=bins)
    probs = counts / max(np.sum(counts), 1)
    return probs


def monitoring_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    workloads = []
    base_d1 = np.array([0.20, 0.80])
    live_d1 = np.array([0.30, 0.70])
    workloads.append({"rung": "D1", "name": "hand baseline and live bins", "base_probs": base_d1, "live_probs": live_d1, "volume": 100})
    for rung, name, volume, shift in [
        ("D2", "clean monitoring stream", 20000, 0.05),
        ("D3", "drift spike with delayed labels", 50000, 0.80),
        ("D4", "breast-cancer-like shifted features", 569, 0.55),
        ("D5", "production feature and prediction stream", 180000, 0.90),
    ]:
        base = local_rng.normal(10.0, 2.0, size=volume)
        live = local_rng.normal(10.0 + shift, 2.0, size=volume)
        pred_base = 1.0 / (1.0 + np.exp(-(base - 10.0) / 2.0))
        pred_live = 1.0 / (1.0 + np.exp(-(live - 10.0) / 2.0))
        bins = np.linspace(4.0, 17.0, 11)
        workloads.append({
            "rung": rung,
            "name": name,
            "base_values": base,
            "live_values": live,
            "base_probs": histogram_probs(base, bins),
            "live_probs": histogram_probs(live, bins),
            "prediction_base": pred_base,
            "prediction_live": pred_live,
            "volume": volume,
        })
    return workloads


## The concept, built once: compare distributions

Population Stability Index is $PSI=\sum_b(p_b-q_b)\ln(p_b/q_b)$ in the lesson notation, with expected $0.20$ and observed $0.30$ producing a single-bin term near $0.0405$. Mean shift is anchored to the baseline contract with $z=(\mu_{live}-\mu_{base})/\sigma_{base}$.

In [ ]:

expected = 0.20
observed = 0.30
psi_bin = (observed - expected) * math.log(observed / expected)
base_mean = 10.0
live_mean = 11.5
base_std = 2.0
z_shift = (live_mean - base_mean) / base_std

assert round(psi_bin, 4) == 0.0405
assert z_shift == 0.75
print("single PSI bin", psi_bin)
print("z shift", z_shift)


The same monitor can also report KS gaps, metric drift, and alert margins. A KS gap from CDF values $0.62$ and $0.47$ is $0.15$; AUC dropping from $0.812$ to $0.781$ is $0.031$; a drift score $0.26$ versus threshold $0.20$ exceeds by $0.06$.

In [ ]:

ks_gap = abs(0.62 - 0.47)
auc_drop = 0.812 - 0.781
alert_margin = 0.26 - 0.20

assert round(ks_gap, 2) == 0.15
assert round(auc_drop, 3) == 0.031
assert round(alert_margin, 2) == 0.06
print("KS gap", ks_gap)
print("AUC drop", auc_drop)
print("alert margin", alert_margin)


## The dataset ladder
D1-D5 move from hand bins to a production-scale feature and prediction stream.

In [ ]:

workloads = monitoring_ladder()
for workload in workloads:
    preview = pd.DataFrame({
        "base_prob": np.round(workload["base_probs"][:5], 4),
        "live_prob": np.round(workload["live_probs"][:5], 4),
    })
    print(workload["rung"], workload["name"], "volume=", workload["volume"])
    print(preview)


In [ ]:

rows = []
outputs = []
for workload in workloads:
    stats = drift_stats(workload["base_probs"], workload["live_probs"], workload.get("base_values"), workload.get("live_values"))
    outputs.append((workload, stats))
    rows.append({
        "rung": workload["rung"],
        "workload": workload["name"],
        "metric_psi": float(stats["psi"]),
        "z_shift": float(stats["z_shift"]) if not np.isnan(stats["z_shift"]) else np.nan,
        "volume": workload["volume"],
    })
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, (workload, stats) in enumerate(outputs):
    x = np.arange(len(workload["base_probs"]))
    axes[0, i].plot(x, workload["base_probs"], marker="o", label="base")
    axes[0, i].plot(x, workload["live_probs"], marker="s", label="live")
    axes[0, i].set_title(workload["rung"])
    axes[0, i].set_xlabel("bin")
    axes[0, i].set_ylabel("probability")
    axes[1, i].bar(["PSI", "volume k"], [stats["psi"], workload["volume"] / 1000.0], color=["#e45756", "#54a24b"])
    axes[1, i].set_title("operational panel")
axes[0, 0].legend()
summary_x = np.arange(len(metrics))
fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary_x, metrics["metric_psi"], marker="o")
ax.axhline(0.20, color="#e45756", linestyle="--", label="alert threshold")
ax.set_xticks(summary_x)
ax.set_xticklabels(metrics["rung"])
ax.set_ylabel("PSI")
ax.set_xlabel("D1 to D5 stream volume/shift")
ax.set_title("drift statistic vs stream")
ax.legend()
plt.show()


## Pitfall on D5: alerting on labels only
Labels arrive late. A label-only monitor can miss a live feature and prediction shift until users have already experienced the changed model behavior.

In [ ]:

d5 = workloads[-1]
label_delay_days = 7
feature_stats = drift_stats(d5["base_probs"], d5["live_probs"], d5["base_values"], d5["live_values"])
pred_bins = np.linspace(0.0, 1.0, 11)
pred_base = histogram_probs(d5["prediction_base"], pred_bins)
pred_live = histogram_probs(d5["prediction_live"], pred_bins)
pred_stats = drift_stats(pred_base, pred_live)
label_only_alert_day = label_delay_days
feature_alert_now = feature_stats["psi"] > 0.20 or pred_stats["psi"] > 0.20
print("label-only alert day", label_only_alert_day)
print("feature PSI", feature_stats["psi"])
print("prediction PSI", pred_stats["psi"])
print("early feature or prediction alert", feature_alert_now)


## Evaluate it + practice

- Metric: PSI drift statistic, with z-shift and delayed-label guardrails.
- No-skill baseline: wait for labels only.
- Cheap sanity check: identical base and live histograms should have PSI near zero.
- Ablation: remove prediction drift monitoring and D5 alerts later.
- Failure signals: PSI driven by tiny bins, live denominator drift, or alert fatigue without ownership.

Practice: Add smoothing to PSI and compare tiny-bin behavior.

Practice: Create a slice monitor for the top 5% predictions.

Practice: Use a different alert threshold and count alert days.